# Protein Pretrain - Windows / Local Notebook

Windows-focused notebook for local Jupyter, but the setup cell also works on Ubuntu/Linux and Colab if the repo is discoverable through `MDNAC_REPO_DIR`, `/content`, or Google Drive.

Assumes `train_part_*.txt` files are already available locally.


In [ ]:
# -- Cross-platform setup: Windows-safe by default --
import os
import sys
from pathlib import Path

def find_repo_dir_for_import(start: Path) -> Path:
    candidates = [start.resolve(), *start.resolve().parents]
    repo_dir_env = os.environ.get("MDNAC_REPO_DIR")
    if repo_dir_env:
        candidates.append(Path(repo_dir_env).expanduser())
    candidates.extend([
        Path("/content/Microbial DNA Compiler"),
        Path("/content/Microbial-DNA-Compiler"),
        Path("/content/drive/MyDrive/Microbial DNA Compiler"),
        Path("/content/drive/MyDrive/Microbial-DNA-Compiler"),
    ])
    for candidate in candidates:
        resolved = candidate.expanduser().resolve()
        if (resolved / "pyproject.toml").exists() and (resolved / "libs").is_dir():
            return resolved
    raise RuntimeError(
        "Could not locate repo. Run inside the repo, set MDNAC_REPO_DIR, "
        "or in Colab clone/mount the repo under /content or Google Drive."
    )


REPO_DIR = find_repo_dir_for_import(Path.cwd())
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from libs.notebook_runtime import bootstrap_notebook, materialize_notebook_training_config
from libs.core.pretrain.training_config import load_protein_training_config
from libs.core.pretrain.protein_lm.trainer import ProteinPretrainTrainer

RUNTIME = bootstrap_notebook(REPO_DIR)
REPO_DIR = Path(RUNTIME["repo_dir"])
CONFIG_SOURCE_PATH = Path(os.environ.get("MDNAC_TRAIN_CONFIG", REPO_DIR / "train.16gb.yaml")).expanduser()
if not CONFIG_SOURCE_PATH.is_absolute():
    CONFIG_SOURCE_PATH = (REPO_DIR / CONFIG_SOURCE_PATH).resolve()
CONFIG_PATH, CONFIG_NOTEBOOK_OVERRIDES = materialize_notebook_training_config(
    CONFIG_SOURCE_PATH,
    project_root=REPO_DIR,
)
config = load_protein_training_config(REPO_DIR, config_path=CONFIG_PATH)

print(f"Setup complete: {REPO_DIR}")
print(f"Effective config: {CONFIG_PATH}")
if CONFIG_NOTEBOOK_OVERRIDES:
    print(f"Notebook overrides: {CONFIG_NOTEBOOK_OVERRIDES}")
RUNTIME


## Verify Local Data

Check that `train_part_*.txt` files exist in the cache directory.

In [ ]:
# ── Verify local train parts ──────────────────────────────────────────────────
cache_dir = Path(config["paths"]["train_part_cache_dir"])
parts = sorted(cache_dir.glob("train_part_*.txt"))

if parts:
    total_size_mb = sum(p.stat().st_size for p in parts) / (1024 * 1024)
    print(f"✅ Found {len(parts)} train_part files ({total_size_mb:.1f} MB total)")
    print(f"   First: {parts[0].name}")
    print(f"   Last:  {parts[-1].name}")

    trainer_probe = ProteinPretrainTrainer(REPO_DIR, config_path=CONFIG_PATH)
    discovered_parts = trainer_probe._discover_local_paths()
    if not discovered_parts:
        raise FileNotFoundError("Trainer could not discover local train_part files.")
    print(f"Trainer will read {len(discovered_parts)} local part file(s)")
    print(f"   Source dir: {discovered_parts[0].parent}")
else:
    # Fallback: check for single train.txt
    train_txt = Path(config["paths"]["train_text_path"])
    if train_txt.exists():
        size_mb = train_txt.stat().st_size / (1024 * 1024)
        print(f"✅ Found single train.txt ({size_mb:.1f} MB)")
    else:
        print(f"❌ No training data found!")
        print(f"   Expected train_part_*.txt in: {cache_dir}")
        print(f"   Or single file at: {train_txt}")
        raise FileNotFoundError("No local training data. Place train_part_*.txt in the cache dir.")

## VRAM Check (16GB)

Before training, verify that the config fits within your GPU's VRAM budget.

In [ ]:
# ── VRAM Preflight Check ──────────────────────────────────────────────────────
from libs.data.training.tokenizer import SequenceTokenizer
from libs.core.pretrain.protein_lm.memory import (
    estimate_protein_pretrain_memory,
    recommend_16gb_train_config,
    _resolve_dtype_from_mixed_precision,
)
from libs.core.pretrain.protein_lm.support.backbone import (
    build_mdc_config_from_progen_config,
    build_progen_config,
)

# Load real tokenizer
tokenizer_map_path = config["paths"]["tokenizer_map_path"]
assert tokenizer_map_path.exists(), f"tokenizer_map.json not found: {tokenizer_map_path}"
tokenizer = SequenceTokenizer.load_map(tokenizer_map_path)
print(f"✅ Tokenizer loaded — vocab_size = {tokenizer.vocab_size}")

# Build model config with mixed precision dtype
mixed_precision = config["runtime"]["mixed_precision"]
resolved_dtype = _resolve_dtype_from_mixed_precision(mixed_precision)
model_cfg = config["model"]
progen_config = build_progen_config(
    model_cfg["progen_model_size"],
    vocab_size=tokenizer.vocab_size,
    context_length=model_cfg["context_length"],
    dtype=resolved_dtype,
)
overrides = model_cfg["progen_config_overrides"]
if overrides:
    progen_config = {**progen_config, **overrides}
model_config = build_mdc_config_from_progen_config(progen_config, dtype=resolved_dtype)

# Estimate VRAM
import torch
estimate = estimate_protein_pretrain_memory(
    model_config=model_config,
    tokenizer=tokenizer,
    batch_size=config["data"]["batch_size"],
    context_length=model_cfg["context_length"],
    optimizer_type=config["optimizer"]["type"],
    dtype=resolved_dtype,
    mixed_precision=mixed_precision,
)

MAX_VRAM_GB = 16.0
TARGET_FRACTION = 0.85
target_budget = min(MAX_VRAM_GB * TARGET_FRACTION, MAX_VRAM_GB - 2.0)

print(f"\n{'='*60}")
print(f"VRAM ESTIMATE (resolved_vocab_size={tokenizer.vocab_size})")
print(f"{'='*60}")
print(f"  Parameters:      {estimate['param_count']:,} ({estimate['param_memory_gb']:.3f} GB)")
print(f"  Gradients:       {estimate['gradient_memory_gb']:.3f} GB")
print(f"  Optimizer state: {estimate['optimizer_state_gb']:.3f} GB")
print(f"  Activations:     {estimate['activation_memory_gb']:.3f} GB")
print(f"  Logits:          {estimate['logits_memory_gb']:.3f} GB")
print(f"  TOTAL:           {estimate['total_estimate_gb']:.3f} GB")
print(f"  Budget:          {target_budget:.3f} GB")
print(f"  Margin:          {target_budget - estimate['total_estimate_gb']:+.3f} GB")

fits = estimate["total_estimate_gb"] <= target_budget
if fits:
    print(f"\n  ✓ Estimated to fit within {MAX_VRAM_GB:.0f}GB")
else:
    print(f"\n  ✗ EXCEEDS {MAX_VRAM_GB:.0f}GB budget!")

if not torch.cuda.is_available():
    print("  ⚠ No CUDA — this is only an estimate. Actual peak may be higher.")

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
trainer = ProteinPretrainTrainer(REPO_DIR, config_path=CONFIG_PATH)
result = trainer.train()
result

In [ ]:
# ── Generate sample ───────────────────────────────────────────────────────────
from libs.core.pretrain.protein_lm.core import generate_protein_text

if trainer.is_main_process and trainer.model is not None:
    sample = generate_protein_text(
        trainer.model,
        trainer.tokenizer,
        prompt="<|protein|>",
        device=trainer.runtime.device,
        max_new_tokens=80,
        context_length=int(trainer.model_config.context_length),
    )
    print(sample)
else:
    print("Sample generation is emitted on rank 0 only.")

In [ ]:
# ── Upload model to HuggingFace (Optional) ────────────────────────────────────
from huggingface_hub import HfApi, login

HF_REPO_ID = "admincybers2/MDNAC-protein-pretrain"  # ← change repo name as needed

login(token=os.environ["HF_TOKEN"])
api = HfApi()

checkpoint_path = result.checkpoint_path
best_path = result.best_checkpoint_path

files_to_upload = []
if checkpoint_path and checkpoint_path.exists():
    files_to_upload.append((str(checkpoint_path), checkpoint_path.name))
if best_path and best_path.exists():
    files_to_upload.append((str(best_path), best_path.name))

# Upload tokenizer_map.json alongside
tokenizer_map = config["paths"]["tokenizer_map_path"]
if Path(tokenizer_map).exists():
    files_to_upload.append((str(tokenizer_map), "tokenizer_map.json"))

# Upload resume_state.json
resume_state = config["paths"]["resume_state_path"]
if Path(resume_state).exists():
    files_to_upload.append((str(resume_state), "resume_state.json"))

for local_path, repo_filename in files_to_upload:
    print(f"Uploading {repo_filename} ...")
    api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=repo_filename,
        repo_id=HF_REPO_ID,
        repo_type="model",
    )

print(f"✅ Uploaded {len(files_to_upload)} file(s) to https://huggingface.co/{HF_REPO_ID}")